In [1]:
import os
import xarray as xr
import s3fs
import fsspec
# this notebook accesses ERA5 climate reanalysis data from AWS S3 storage, calculates and saves
# the average and maximum summer (June, July, August) temperature in the grid cell containing
# Brownsville, Brooklyn - motivating the argument that public housing in this location was designed
# for a different climate than the one we now inhabit

In [2]:
OSN_ENDPOINT_URL = "https://nyu1.osn.mghpcc.org"
OSN_BUCKET = "leap-pangeo-manual"
HACKATHON_PREFIX = "hackathon-2026/"

In [3]:
OSN_ROOT = f"s3://{OSN_BUCKET}/{HACKATHON_PREFIX}" 

# For this hackathon: s3://leap-pangeo-manual/hackathon-2026/

In [4]:
# Anonymous access to a public OSN endpoint
fs_osn = s3fs.S3FileSystem(
    anon=True,
    client_kwargs={"endpoint_url": OSN_ENDPOINT_URL},
)

# Test: list top-level within the hackathon prefix
fs_osn.ls(f"{OSN_BUCKET}/{HACKATHON_PREFIX}")[:20]

['leap-pangeo-manual/hackathon-2026/corrdiff',
 'leap-pangeo-manual/hackathon-2026/era5_cds',
 'leap-pangeo-manual/hackathon-2026/hrrr']

In [5]:
fs_osn.ls(f"{OSN_BUCKET}/{HACKATHON_PREFIX}era5_cds/")[:20]

['leap-pangeo-manual/hackathon-2026/era5_cds/NYC']

In [6]:
fs_osn.ls(f"{OSN_BUCKET}/{HACKATHON_PREFIX}era5_cds/NYC/")[:20]

['leap-pangeo-manual/hackathon-2026/era5_cds/NYC/era5NYC1945.zarr',
 'leap-pangeo-manual/hackathon-2026/era5_cds/NYC/era5NYC1955.zarr',
 'leap-pangeo-manual/hackathon-2026/era5_cds/NYC/era5NYC1965.zarr',
 'leap-pangeo-manual/hackathon-2026/era5_cds/NYC/era5NYC1975.zarr',
 'leap-pangeo-manual/hackathon-2026/era5_cds/NYC/era5NYC1985.zarr',
 'leap-pangeo-manual/hackathon-2026/era5_cds/NYC/era5NYC1995.zarr',
 'leap-pangeo-manual/hackathon-2026/era5_cds/NYC/era5NYC2005.zarr',
 'leap-pangeo-manual/hackathon-2026/era5_cds/NYC/era5NYC2012.zarr',
 'leap-pangeo-manual/hackathon-2026/era5_cds/NYC/era5NYC2015.zarr',
 'leap-pangeo-manual/hackathon-2026/era5_cds/NYC/era5NYC2019.zarr',
 'leap-pangeo-manual/hackathon-2026/era5_cds/NYC/era5NYC2020.zarr',
 'leap-pangeo-manual/hackathon-2026/era5_cds/NYC/era5NYC2021.zarr',
 'leap-pangeo-manual/hackathon-2026/era5_cds/NYC/era5NYC2022.zarr',
 'leap-pangeo-manual/hackathon-2026/era5_cds/NYC/era5NYC2023.zarr',
 'leap-pangeo-manual/hackathon-2026/era5_cds/NYC

In [7]:
years=["1945","1955","1965","1975","1985","1995","2005","2015","2025"]

In [9]:
for year in years:
    example_zarr_path = f"{OSN_BUCKET}/{HACKATHON_PREFIX}era5_cds/NYC/era5NYC{year}.zarr"
    mapper = fs_osn.get_mapper(example_zarr_path)
    ds = xr.open_zarr(mapper, consolidated=False)  # consolidated=True if metadata consolidated
    #select Brownsville grid cell from ERA5 data
    ds_bv=ds.sel(latitude=40.66,longitude=-73.91,method='nearest')
    #select June-July-August data
    ds_bv_jja=ds_bv.isel(time=slice(151*24,243*24)) #151 days prior to June 1 x 24 hours, etc...
    #select temperature variable
    temp=ds_bv_jja['t2m'].values
    #print to screen mean and max values to check conversion from Kelvin to Fahrenheit
    print(year, (temp.mean()-273.15)*(9./5.)+32., (temp.max()-273.15)*(9./5.)+32.)
    #write same output as csv
    with open('data_out.csv', 'a') as csvfile:
        csvfile.write(str(year) + "," + str((temp.mean()-273.15)*(9./5.)+32.) + "," + str((temp.max()-273.15)*(9./5.)+32.) + '\n')

1945 70.66842828110768 88.44259342040108
1955 73.23996276765496 91.34355797723101
1965 70.94787231853786 85.96193325771347
1975 73.04039024805692 92.03720210112425
1985 71.56368765417344 87.71614217905366
1995 73.52171893392227 92.94986566450186
2005 74.30092188376685 90.84446331860596
2015 73.74644615196038 86.80544091574097
2025 73.99584372940348 94.02911158190346
